# ML-02 — Research Question and Provisional Lane

**Lane:** Freestyle — Growth / Recovery / Momentum Prediction (FlyRank ML Internship)
**Owner:** Michael Adesiyan
**Research Question:** Can we predict which pages are likely to decline, recover, or gain momentum?

## 1. My lane (or freestyle) and why

**Lane Choice:** Freestyle — Growth / Recovery / Momentum Prediction

**Why this lane?**
In SEO content strategy and website operations, managing existing content is often more high-leverage than creating new content from scratch. Identifying declining pages early allows editors to intervene with targeted refreshes before search visibility and organic traffic disappear completely. Conversely, recognizing pages early as they gain momentum allows publishing teams to double down on winning content (e.g. adding internal links or expanding topic depth).

Predicting momentum, recovery, and decline enables a proactive, ranked queue for content operations rather than reactive troubleshooting.

In [1]:
# Lane & Core Label Framing Setup
LANE_NAME = "Freestyle — Growth / Recovery / Momentum Prediction"
FEATURE_WINDOW_DAYS = 90  # Prior 90 days of features
TARGET_WINDOW_DAYS = 30   # Next 30 days observed outcome

print(f"Lane Selected: {LANE_NAME}")
print(f"Feature Window: Prior {FEATURE_WINDOW_DAYS} days -> Target Window: Next {TARGET_WINDOW_DAYS} days")
print("Golden Rule: Feature window must NEVER overlap target window.")

Lane Selected: Freestyle — Growth / Recovery / Momentum Prediction
Feature Window: Prior 90 days -> Target Window: Next 30 days
Golden Rule: Feature window must NEVER overlap target window.


## 2. The question: decision, action, cost of a wrong call

### Framing Answers:

1. **What decision does this improve?**
   - It improves content operational resource allocation: deciding *which specific URLs* an editorial team should review, refresh, or optimize first to prevent traffic loss or accelerate momentum.

2. **Who acts on the output, and what do they do?**
   - **Actors:** Content Managers, SEO Specialists, and Editorial Teams.
   - **Action:** Scheduling content refreshes, updating outdated figures/sources, improving internal link structure, or adding fresh sections to high-momentum pages.

3. **What does a wrong answer cost?**
   - **False Positive (recommending a stable/growing page for urgent refresh):** Wasted editor/writer hours and unnecessary content changes that could destabilize ranking.
   - **False Negative (missing a page undergoing steep decline):** Permanent loss of organic search traffic, lower conversions/revenue, and diminished domain authority.

4. **Why does ML help at all?**
   - Plain rules (e.g., "refresh every page older than 180 days") are noisy and ignore multi-signal interactions across GSC impressions, GA4 engagement, keyword position shifts, and historical velocity.
   - ML earns its place by modeling non-linear signal combinations across prior feature windows to score future decline/momentum probabilities accurately.

### ML Mapping:
- **Task Type:** Ranking / Scoring (Priority Queue) + Classification (Decline vs Growth/Stable probability)
- **Target:** Observed outcome in future target window (e.g., observed 30-day post-window impression/traffic change)
- **Primary Metric:**  (e.g., ,  matching weekly review capacity) and .

In [2]:
# Define primary metrics & target shape
PRIMARY_METRIC = "Precision@50"
SECONDARY_METRIC = "ROC-AUC"
EXCLUDED_FIELDS = ["health_score", "priority_score", "action_type"]

print(f"Primary Evaluation Metric: {PRIMARY_METRIC}")
print(f"Secondary Metric: {SECONDARY_METRIC}")
print(f"Explicitly Excluded Product Labels/Proxies: {EXCLUDED_FIELDS}")

Primary Evaluation Metric: Precision@50
Secondary Metric: ROC-AUC
Explicitly Excluded Product Labels/Proxies: ['health_score', 'priority_score', 'action_type']


## 3. Quick look at the data (2-3 real numbers)

Let's load the starter dataset () and compute foundational numbers to validate our lane.

In [3]:
import os
import pandas as pd
import numpy as np

# Load starter dataset dynamically based on working directory
csv_path = "data/raw/content_refresh_anonymized.csv" if os.path.exists("data/raw/content_refresh_anonymized.csv") else "../../data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(csv_path)

# Number 1: Dataset scale & breadth
n_rows, n_cols = df.shape
n_clients = df["client_id"].nunique()
print(f"[Real Number 1] Dataset Scale: {n_rows:,} content items across {n_clients} anonymized clients ({n_cols} columns).")

# Number 2: Baseline decline rate & trend distribution
trend_counts = df["trend_direction"].value_counts()
trend_pcts = df["trend_direction"].value_counts(normalize=True) * 100
down_rate = trend_pcts.get("down", 0.0)
print(f"[Real Number 2] Baseline Trend Distribution:")
for direction, pct in trend_pcts.items():
    print(f"  - {direction}: {trend_counts[direction]:,} rows ({pct:.2f}%)")
print(f"  -> Declining Content Baseline Rate: {down_rate:.2f}%")

# Number 3: Key data gotchas (Missing search position indicator)
no_position_count = (df["avg_position"] == 0).sum()
no_position_pct = (df["avg_position"] == 0).mean() * 100
print(f"[Real Number 3] Data Gotcha Check (avg_position == 0 represents 'no data'):")
print(f"  - Rows with avg_position == 0: {no_position_count:,} ({no_position_pct:.2f}%)")
print(f"  - Active search visibility (impressions_90d > 0): {(df['impressions_90d'] > 0).sum():,} rows ({(df['impressions_90d'] > 0).mean()*100:.2f}%).")


[Real Number 1] Dataset Scale: 30,000 content items across 32 anonymized clients (44 columns).
[Real Number 2] Baseline Trend Distribution:
  - down: 16,262 rows (54.21%)
  - stable: 5,962 rows (19.87%)
  - up: 4,388 rows (14.63%)
  - new: 2,236 rows (7.45%)
  - flat: 1,152 rows (3.84%)
  -> Declining Content Baseline Rate: 54.21%
[Real Number 3] Data Gotcha Check (avg_position == 0 represents 'no data'):
  - Rows with avg_position == 0: 1,205 (4.02%)
  - Active search visibility (impressions_90d > 0): 30,000 rows (100.00%).


## 4. Careful words: what I can and can't claim

### What this work CAN claim:
- **Observed Historical Relationships:** Signal correlations between historical activity (GSC/GA4) and subsequent 30-day performance changes.
- **Directional Risk & Opportunity Scores:** Probability scores that rank pages by risk of future decline or potential for momentum out-of-sample.
- **Decision-Support Utility:** Demonstrating that an ML-based ranking model outperforms plain heuristic baselines on  in validation holdouts.

### What this work CANNOT claim:
- **Causal Proof:** We cannot claim that performing a content refresh *caused* a recovery without a randomized controlled experiment (A/B testing).
- **Algorithm Decoding:** We do not claim to "predict Google's algorithm" or reverse-engineer search engine ranking formulas.
- **Deterministic Traffic Guarantees:** Predictions are probabilistic decision support tools, not deterministic guarantees of future rankings.

In [4]:
# Summary statement of claims
print("=== Scope & Claims Boundaries ===")
print("CLAIM TYPE: Directional & Decision-Support Ranking")
print("EVALUATION BOUNDARY: Out-of-sample / Client-grouped validation")
print("EXPLICIT NON-CLAIMS: No causal attribution, No algorithm reverse-engineering")

=== Scope & Claims Boundaries ===
CLAIM TYPE: Directional & Decision-Support Ranking
EVALUATION BOUNDARY: Out-of-sample / Client-grouped validation
EXPLICIT NON-CLAIMS: No causal attribution, No algorithm reverse-engineering


## Self-check

Before submitting, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under  — ready for submission.